#### install library

In [ ]:
!pip install pygame

#### import library

In [2]:
# import library

import pygame as pyg
import numpy as np
import pandas as pd
import math
import time

from pygame.locals import *
from pygame.rect import *

pygame 2.0.1 (SDL 2.0.14, Python 3.8.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
# Set the color : pyg.Color(Red value, Green value, Blue value)

RED = pyg.Color(255, 0, 0)
GRAY = pyg.Color(128, 128, 128)
GREEN = pyg.Color(0,255,0)
WHITE = pyg.Color(255,255,255)
BLACK = pyg.Color(0,0,0)
BLUE = pyg.Color(0,0,255)

# Set the display parameters : width, height

width, height = 1440, 850

#### Define Environment Class

In [4]:
class Environment:
    
    def __init__(self, width, height):
        self.running = True 
        self.clock = pyg.time.Clock() 
        self.duration = 60 * 24
        self.width = width
        self.height = height
        self.count = 0 

        
        self.Screen = Screen(self.width, self.height) 
        self.Target = Target(width = self.width, height = self.height, cursor_diameter = 20, target_diameter = 70) 
        self.Joystick = Joystick()
        
        max_x, max_y = self.width - self.Target.diameter_1, self.height - self.Target.diameter_1
        self.Mode = Mode(max_x = max_x, max_y = max_y)
        
    def run(self):
        
        pyg.init() # pygame start
        self.Screen.open_console() # open the console window
        self.Target.initialize_rect() # Make a target.
        
        i = 0
        
        while self.running:
            
            # This loop functions as a flag to close the pygame. 
            for event in pyg.event.get():
                if event.type == QUIT:
                    self.running = False
            
            pyg.event.pump() 
            
            self.clock.tick(60) # Set the clock as 60Hz.
            self.Screen.over_write() # Set the console window's background color. default = BLACK
            gradient = self.Target.move() # Make Target moving.
            
            index = ['c','a','b','a','b','c','b','a','b','a']
            
            if index[i] == 'c':
                
                target, x, y = self.Mode.base(self.Joystick.joystick, self.Screen.screen, self.Target.rect)
                
            elif index[i] == 'a':
                
                target, x, y = self.Mode.adaptation(self.Joystick.joystick, self.Screen.screen, self.Target.rect)
                
            else:    
                
                target, x, y = self.Mode.reverse(self.Joystick.joystick, self.Screen.screen, self.Target.rect, gradient)
                
            self.Target.cursor(self.Screen.screen, x, y)
            target_pos = list(self.Target.current_pos)
            
            if x >=  target_pos[0] and x <=  target_pos[0] + self.Target.diameter_2 and y >=  target_pos[1] and y <=  target_pos[1] + self.Target.diameter_2:
                
                self.Target.hit(self.Screen.screen)
    
            self.count += 1
            self.Screen.flip()
            
            if self.count % 1440 == 0:
                i +=1
                if i == 10:
                    i = 0
                
            
        pyg.quit()

#### Define Screen Class

In [5]:
class Screen:

    '''
    Class
    
        Name : Screen
        Function : 1) Set the console window 
                   2) Flip the console window
    
    Method
    
        1) open_console
        
         - This function will make a console window 
           which width and heights can be set by a experimenter. 
        
         - Parameters : width, height
         - default : width = 1440, height = 850
        
        2) over_write
        
        - This function will set the color of the display.
        
        - Parameters : color
        - default : color = BLACK 
        
        3) flip
        
        - This function will update the display.
        
        - Parameters : None
    
    '''
    
    def __init__(self, width, height):
        
        self.width = width
        self.height = height
    
    def open_console(self,):
        
        self.size = self.width, self.height
        self.screen = pyg.display.set_mode(self.size)
        
    def over_write(self, color = BLACK):
        
        self.screen.fill(color)
    
    def flip(self):
        
        pyg.display.flip()    

#### Define Target Class

In [6]:
class Target:
    
    '''
    Class
    
        Name : Target
        Function : 1) Display a target & cursor
                   2) Set the modes of target & cursor 
    
    Method
    
        1) move
        
        - This move method is responsible to move target.
          In the pygame library, you can set the values of aixs x and y.
          
        - parameters : speed ( type : list)
        - default : speed = [1, 1]
        
        2) cursor
        
        - This cursor method will make an circle for cursor.
          Cursor will move based on x and y
        
        - parameters : x, y
        
        3) initialize_rect
        
        - This method is responsible to target's size.
        
        - Rect(x, y, width, height)
        
        4) hit
        
        - When the cursor is in the scope of target, the target will be white.
        
    '''
    
    def __init__(self, width, height, cursor_diameter = 20, target_diameter = 70):
        
        self.width = width
        self.height = height
        self.diameter_1 = cursor_diameter
        self.diameter_2 = target_diameter
        
    def move(self, speed = [1, 1]): 
        
        self.speed = speed
        self.rect.move_ip(self.speed)
        self.current_pos = self.rect.copy()
        
        if self.rect.left < 0:
            self.speed[0] *= -1
        if self.rect.right > self.width:
            self.speed[0] *= -1
        if self.rect.top < 0:
            self.speed[1] *= -1
        if self.rect.bottom > self.height:
            self.speed[1] *= -1
        
        return self.speed[1] / self.speed[0]
        
    def cursor(self,screen, x , y):
        pyg.draw.ellipse(screen, GREEN, Rect(x, y, self.diameter_1, self.diameter_1), width = 3)
        
    def initialize_rect(self):
        self.rect = Rect(500, 500, self.diameter_2, self.diameter_2) # rect의 ( x좌표 , y좌표, 폭, 높이 )
        
    def hit(self,screen):
        pyg.draw.ellipse(screen, WHITE, self.rect, width = 8)

In [7]:
class Joystick:
    
    
    '''
    Class
    
        Name : Joystick
        Function : 1) Set the joystick environment
    
    Method
    
        1) print(Joystick) : print values of Joystick which are relevant to axis x and y.
        
    '''
    
    def __init__(self):
        
        pyg.joystick.init()
        self.joystick = pyg.joystick.Joystick(0)
        self.joystick.init()
        print("Joystick enable : ", self.joystick.get_init())
    
    def __str__(self):
        
        return 'Axis x : {:.3f}, Axis y : {:.3f}'.format(self.joystick.get_axis(0),self.joystick.get_axis(1))

In [8]:
class Mode:
    
    
    '''
    Class
    
        Name : Mode
        Function : 1) Display a target which is related to a perturbation.
                   2) Set the modes of cursor 
    
    Method
    
        1) base
        
        - This base method is responsible to move cursor.
        
        - perturbation : no
        
        
        2) adaptation
        
        - This adaptation method is responsible to move cursor.
        
        - perturbation : adaptation ( 60 degree)
        
        3) reverse
        
        - This reverse method is responsible to move cursor.
        
        - perturbation : reverse
        
        
        
    '''
    
    def __init__(self, max_x, max_y):
        
        self.curr_x = 700
        self.curr_y = 500
        self.max_x = max_x
        self.max_y = max_y
    
    
    def base(self, joystick, screen, rect):
        
        self.target = pyg.draw.ellipse(screen, GRAY, rect, width = 8)
        
        prev_x = joystick.get_axis(0) * 2
        prev_y = joystick.get_axis(1) * 2
        
        self.curr_x += prev_x
        self.curr_y += prev_y
        
        if self.curr_y <=0:
            self.curr_y = 0
        elif self.curr_y >= self.max_y:
            self.curr_y = self.max_y
            
        if self.curr_x <=0:
            self.curr_x = 0
        elif self.curr_x >= self.max_x:
            self.curr_x = self.max_x
        
        return self.target, self.curr_x, self.curr_y
        
        
    def adaptation(self, joystick, screen, rect, degree = 60):
        
        self.target = pyg.draw.ellipse(screen, BLUE, rect, width = 8)
        self.degree = degree
        
        prev_x = joystick.get_axis(0) * 2
        prev_y = joystick.get_axis(1) * 2
        
        prev_x_rot = np.cos(np.deg2rad(self.degree))*prev_x - np.sin(np.deg2rad(self.degree))*prev_y
        prev_y_rot = np.sin(np.deg2rad(self.degree))*prev_x + np.cos(np.deg2rad(self.degree))*prev_y
        
        self.curr_x += prev_x_rot
        self.curr_y += prev_y_rot
        
        if self.curr_y <=0:
            self.curr_y = 0
        elif self.curr_y >= self.max_y:
            self.curr_y = self.max_y
            
        if self.curr_x <=0:
            self.curr_x = 0
        elif self.curr_x >= self.max_x:
            self.curr_x = self.max_x
        
        
        return self.target, self.curr_x, self.curr_y
    
    
    def reverse(self, joystick, screen, rect, gradient, degree = 30):
        self.target = pyg.draw.ellipse(screen, RED, rect, width = 8)
        self.degree = degree
        
        prev_x = joystick.get_axis(0) * 4
        prev_y = joystick.get_axis(1) * 4
        
        degree_2 = - np.rad2deg(np.arctan(prev_y/prev_x))
        
        
        if prev_x != 0:
            
            if gradient <= 0:
                
                if prev_x > 0:
                    
                    theta = 45 - degree_2 + self.degree
                    theta_final = 45 + self.degree + theta
                    
#                     if degree_2 >= 0:
#                         print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2, theta, theta_final))
#                     else:
#                         print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 360, theta, theta_final))
                    
                elif prev_x < 0:
                        
                    theta = 180 + degree_2 - 45 - self.degree
                    theta_final = 45 + self.degree - theta 
                    
#                     if degree_2 >= 0:
                    
#                         if theta_final > 0:
#                             print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 180, theta, theta_final))
#                         elif theta_final < 0:
#                             print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 180, theta, theta_final + 360))
#                     else:
#                         if theta_final > 0:
#                             print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 180, theta, theta_final))
#                         elif theta_final < 0:
#                             print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 180, theta, theta_final + 360))
                    
            elif gradient > 0:
                
                if prev_x > 0:
                    
                    theta = 135 - degree_2 + self.degree
                    theta_final = 135 + self.degree + theta
                    
#                     if degree_2 >= 0:
#                         print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2, theta, theta_final % 360))
#                     else:
#                         print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 360, theta, theta_final% 360))
                    
                    
                elif prev_x < 0:
                        
                    theta = 180 - 135 + degree_2 - self.degree
                    theta_final = 135 + self.degree - theta

#                     if degree_2 >= 0:              
#                         print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 180, theta, (theta_final + 360)%360))
#                     else:
#                         print('cursor_degree : {:.2f}, theta : {:.2f}, display_degree : {:.2f}'.format(degree_2 + 180, theta, (theta_final + 360)%360))


                    
        else:
            
            if gradient <= 0:
                
                if prev_y >= 0:
                    theta_final = 60
                    
                elif prev_y < 0:
                    theta_final = 240
                    
            elif gradient > 0:
                
                if prev_y >= 0:
                    theta_final = 240
                    
                elif prev_y < 0:
                    theta_final = 60
        
        theta_final = -theta_final
        
        prev_x_rev = np.sqrt(prev_x ** 2 + prev_y ** 2) * np.cos(np.deg2rad(theta_final))
        prev_y_rev = np.sqrt(prev_x ** 2 + prev_y ** 2) * np.sin(np.deg2rad(theta_final))
        
        self.curr_x += prev_x_rev
        self.curr_y += prev_y_rev
            
        if self.curr_y <=0:
            self.curr_y = 0
        elif self.curr_y >=self.max_y:
            self.curr_y = self.max_y
            
        if self.curr_x <=0:
            self.curr_x = 0
        elif self.curr_x >= self.max_x:
            self.curr_x = self.max_x
        
        return self.target, self.curr_x, self.curr_y

In [9]:
env = Environment(width, height)

Joystick enable :  True


In [10]:
env.run()

2021-07-14 13:54:04.727 python[6473:74767] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit
